In [ ]:
import os


path_images = "data/loveda/val/Rural/images_png"
image = "1.png"
path_image = os.path.join(path_images, image).replace("\\", "/")
print(path_image)

data/loveda/val/Rural/images_png/1.png


In [5]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: For pretty scientific plots
try:
    import scienceplots
    plt.style.use(["science", "grid", "high-vis", "no-latex"])
except ImportError:
    plt.style.use("default")

Imports and Setup

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Adjust these paths as needed
sys.path.append('path/to/your/repo')  # so you can import datasets
from datasets.loveda import LoveDA

# LoveDA 7-class color palette (from your code)
LOVEDA_COLORMAP = np.array([
    [255, 255, 255],  # 0: Background
    [255, 0, 0],      # 1: Building
    [0, 255, 0],      # 2: Road
    [0, 0, 255],      # 3: Water
    [255, 255, 0],    # 4: Barren
    [0, 255, 255],    # 5: Forest
    [255, 0, 255],    # 6: Agriculture
], dtype=np.uint8)

ModuleNotFoundError: No module named 'cv2'

Instantiate the Dataset

In [ ]:
# Set these paths to your actual data locations
root = 'data/'  # or wherever your LoveDA root is
list_path = 'list/loveda/val.lst'  # or train.lst, etc.

dataset = LoveDA(
    root=root,
    list_path=list_path,
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=-1,
    base_size=1024,
    crop_size=(1024, 1024)
)

print(f"Dataset length: {len(dataset)}")

Visualize a Few Samples and Print Label Ranges

In [ ]:
def decode_segmap(mask, colormap=LOVEDA_COLORMAP, ignore_index=None):
    if mask.ndim == 2:
        mask = mask[None, ...]
    N, H, W = mask.shape
    color_masks = np.zeros((N, H, W, 3), dtype=np.uint8)
    for i in range(N):
        for cls_idx, color in enumerate(colormap):
            color_masks[i][mask[i] == cls_idx] = color
        if ignore_index is not None:
            color_masks[i][mask[i] == ignore_index] = [0, 0, 0]
    if color_masks.shape[0] == 1:
        return color_masks[0]
    return color_masks

num_samples = 5
all_labels = []

for idx in range(num_samples):
    image, label, edge, size, name = dataset[idx]
    # image: torch.Tensor (C, H, W), label: torch.Tensor (H, W)
    img_np = image.numpy().transpose(1,2,0)
    # Unnormalize if needed (your dataset may already normalize)
    img_np = np.clip(img_np, 0, 1)
    label_np = label.numpy()
    all_labels.append(label_np)
    color_mask = decode_segmap(label_np)
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.imshow(img_np)
    plt.title(f"Image: {name}")
    plt.axis('off')
    plt.subplot(1,2,2)
    plt.imshow(color_mask)
    plt.title(f"Label (colored): {name}")
    plt.axis('off')
    plt.show()
    print(f"Label unique values for {name}: {np.unique(label_np)}")

# Flatten all labels and print global min/max/unique
all_labels_flat = np.concatenate([l.flatten() for l in all_labels])
print(f"\nGlobal label value range in {num_samples} samples: {all_labels_flat.min()} to {all_labels_flat.max()}")
print(f"Unique label values in {num_samples} samples: {np.unique(all_labels_flat)}")

Histogram of Label Distribution

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(all_labels_flat, bins=np.arange(-1,8)-0.5, rwidth=0.8)
plt.title("Label Value Distribution in Sampled Masks")
plt.xlabel("Label Value")
plt.ylabel("Pixel Count")
plt.xticks(np.arange(-1,7))
plt.show()